In [1]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q datasets
!pip install -q accelerate
!pip install -q scikit-learn
!pip install -q transformers qwen-vl-utils
!pip install evaluate rouge_score bert_score scikit-learn
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 31.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 62.6 MB/s eta 0:00:00:00:0100:01
✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!


In [2]:
import os
import gc
import re
import copy
import time
import urllib.request
import zipfile
import torch
import numpy as np
import pandas as pd
from PIL import Image
from datasets import load_dataset
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# ==========================================
# 1. CÔNG CỤ ĐO LƯỜNG VÀ QUẢN LÝ BỘ NHỚ
# ==========================================
def clearMemory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def getPeakReservedMemory():
    if torch.cuda.is_available():
        reservedGb = torch.cuda.memory_reserved() / (1024 ** 3)
        return f"{reservedGb:.2f} GB"
    return "0.00 GB"

def calculateTrainableParameters(model):
    trainableParams = 0
    allParams = 0
    for _, param in model.named_parameters():
        allParams += param.numel()
        if param.requires_grad:
            trainableParams += param.numel()
    percentage = 100 * trainableParams / allParams
    return trainableParams, allParams, percentage

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH TOÀN CỤC (CENTRALIZED)
# ==========================================
print("--- KHỞI TẠO HẠ TẦNG CENTRALIZED LORA ---")
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28 
)

baseModel = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map={"": 0} 
)

baseModel.config.use_cache = False
baseModel.gradient_checkpointing_enable()

loraConfig = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
    bias="none"
)

centralizedModel = get_peft_model(baseModel, loraConfig)
trainableParams, totalModelParams, trainPct = calculateTrainableParameters(centralizedModel)
print(f"[Metrics] Trainable Params: {trainableParams:,} (~{trainPct:.3f}%)")

# ==========================================
# 3. CHUẨN BỊ DỮ LIỆU SLAKE (GỘP CHUNG 1 TẬP)
# ==========================================
print("\n--- CHUẨN BỊ DỮ LIỆU SLAKE (FULL CENTRALIZED ENGLISH DATASET) ---")

if not os.path.exists("./imgs"):
    print(" [Hệ thống] Đang tải file ảnh imgs.zip từ Hugging Face...")
    urllib.request.urlretrieve("https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip", "imgs.zip")
    with zipfile.ZipFile("imgs.zip", 'r') as zipRef:
        zipRef.extractall("./")
    os.remove("imgs.zip")
    print(" [Hệ thống] Đã chuẩn bị xong thư mục ảnh cục bộ!")

dataset = load_dataset("BoKelvin/SLAKE")
englishTrain = dataset["train"].filter(lambda x: x["q_lang"] == "en")
englishTest = dataset["test"].filter(lambda x: x["q_lang"] == "en")

trainDataset = englishTrain.shuffle(seed=42)
testDataset = englishTest

print(f"Dữ liệu Tập Train (Tiếng Anh): {len(trainDataset)} mẫu")
print(f"Dữ liệu Tập Test (Tiếng Anh): {len(testDataset)} mẫu")

headerTokens = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
headerLen = len(headerTokens)

# ==========================================
# CẤU TRÚC PROMPT TỐI ƯU
# ==========================================
systemInstruction = (
    "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
    "based EXCLUSIVELY on the provided medical image.\n"
    "Strictly follow these rules:\n"
    "1. Analyze the provided [Target Case Image] and the [Question].\n"
    "2. Provide a concise and definitive answer (1-3 words if possible)."
)

def customCollateFn(batch):
    texts = []
    images = []
    
    for item in batch:
        imgName = item["img_name"].replace("imgs/", "")
        imgPath = os.path.join("./imgs", imgName)
        img = Image.open(imgPath).convert("RGB")
        
        q = item["question"]
        a = str(item["answer"]).strip()
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": systemInstruction}]},
            {"role": "user", "content": [
                {"type": "text", "text": "[Target Case Image]\n"}, 
                {"type": "image", "image": img}, 
                {"type": "text", "text": f"\nCurrent Question: {q}\n\nBased on the visual evidence, what is the correct answer for the current case?\nFinal Answer:"}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]}
        ]
        
        textPrompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(textPrompt)
        images.append(img)

    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = inputs["input_ids"].clone()
    
    for i in range(labels.shape[0]):
        tokenIds = labels[i].tolist()
        if processor.tokenizer.pad_token_id is not None:
            labels[i][labels[i] == processor.tokenizer.pad_token_id] = -100
            
        matchIdx = -1
        for j in range(len(tokenIds) - headerLen + 1):
            if tokenIds[j:j+headerLen] == headerTokens:
                matchIdx = j + headerLen 
                break
        if matchIdx != -1:
            labels[i, :matchIdx] = -100 

    inputs["labels"] = labels
    return inputs

batchSize = 2
trainLoader = DataLoader(trainDataset, batch_size=batchSize, shuffle=True, collate_fn=customCollateFn, num_workers=2, pin_memory=True)

# ==========================================
# 4. HÀM ĐÁNH GIÁ (ĐÃ TỐI ƯU CLOSED QUESTIONS)
# ==========================================
def cleanOpenAnswer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    stopwords = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

def evaluateModel(model, dataset, processor):
    model.eval()
    print(f"\n [Hệ thống] Đang đánh giá trên tập Test ({len(dataset)} mẫu)...")
    
    openPreds, openRefs = [], []
    closePreds, closeRefs = [], []
    overallCorrect = 0
    inferenceTimes = []
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    for i, item in enumerate(dataset):
        imgName = item["img_name"].replace("imgs/", "")
        imgPath = os.path.join("./imgs", imgName)
        img = Image.open(imgPath).convert("RGB")
        
        q = item["question"]
        gtAnswer = str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "")).strip().upper()
        if ansType == "":
            ansType = "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN"
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": systemInstruction}]},
            {"role": "user", "content": [
                {"type": "text", "text": "[Target Case Image]\n"}, 
                {"type": "image", "image": img}, 
                {"type": "text", "text": f"\nCurrent Question: {q}\n\nBased on the visual evidence, what is the correct answer for the current case?\nFinal Answer:"}
            ]}
        ]
        
        textPrompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[textPrompt], images=[img], return_tensors="pt", padding=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        startTime = time.time()
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
            generatedIds = model.generate(**inputs, max_new_tokens=15, do_sample=False)
        inferenceTimes.append(time.time() - startTime)
            
        generatedIdsTrimmed = [outIds[len(inIds):] for inIds, outIds in zip(inputs["input_ids"], generatedIds)]
        aiAnswerRaw = processor.batch_decode(generatedIdsTrimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip().lower()
        aiAnswer = aiAnswerRaw.replace("final answer:", "").strip()
        
        aiAnswerClean = cleanOpenAnswer(aiAnswer)
        gtAnswerClean = cleanOpenAnswer(gtAnswer)
        
        if ansType == "CLOSED":
            if gtAnswer in ['yes', 'no']:
                hasYes = any(word in aiAnswer for word in yesKeywords)
                hasNo = any(word in aiAnswer for word in noKeywords)
                
                if hasYes and not hasNo:
                    predClean = 'yes'
                elif hasNo and not hasYes:
                    predClean = 'no'
                else:
                    firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                    firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                    predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            else:
                predClean = aiAnswerClean
                
            targetGt = gtAnswerClean if gtAnswer not in ['yes', 'no'] else gtAnswer
            closePreds.append(predClean)
            closeRefs.append(targetGt)
            
            if predClean == targetGt:
                overallCorrect += 1
        else:
            openPreds.append(aiAnswerClean)
            openRefs.append(gtAnswerClean)
            
            if aiAnswerClean == gtAnswerClean:
                overallCorrect += 1

        del inputs, generatedIds, generatedIdsTrimmed
        torch.cuda.empty_cache()

    results = {
        "overallAccuracy": overallCorrect / len(dataset) if len(dataset) > 0 else 0,
        "avgInferenceTime": sum(inferenceTimes) / len(inferenceTimes)
    }
    
    if len(closeRefs) > 0:
        acc = accuracy_score(closeRefs, closePreds)
        f1Weighted = f1_score(closeRefs, closePreds, average='weighted', zero_division=0)
        results["Closed"] = {"ACC": acc, "F1": f1Weighted, "count": len(closeRefs)}
    
    if len(openRefs) > 0:
        bleuMetric = evaluate.load("bleu")
        rougeMetric = evaluate.load("rouge")
        bertscoreMetric = evaluate.load("bertscore")
        
        formattedRefs = [[ref] for ref in openRefs] 
        bleuRes = bleuMetric.compute(predictions=openPreds, references=formattedRefs, max_order=1)
        rougeRes = rougeMetric.compute(predictions=openPreds, references=formattedRefs)
        bertRes = bertscoreMetric.compute(predictions=openPreds, references=openRefs, lang="en")
        
        results["Open"] = {
            "BLEU-1": bleuRes["bleu"],
            "ROUGE-L": rougeRes["rougeL"],
            "BERTScore": sum(bertRes["f1"]) / len(bertRes["f1"]),
            "count": len(openRefs)
        }
        
    gc.collect()
    return results

# ==========================================
# 5. VÒNG LẶP HUẤN LUYỆN TẬP TRUNG (2 EPOCHS)
# ==========================================
totalEpochs = 2  # Đã chỉnh về 2 Epochs đồng bộ với các bản Federated
learningRate = 5e-5

optimizer = torch.optim.AdamW(centralizedModel.parameters(), lr=learningRate)
performanceLog = []
peakVramRecorded = "0.00 GB"

print(f"\n=== BẮT ĐẦU HUẤN LUYỆN TẬP TRUNG (CENTRALIZED LORA - {totalEpochs} EPOCHS) ===")
startTotalTime = time.time()

for epoch in range(totalEpochs):
    centralizedModel.train()
    print(f"\n--- EPOCH {epoch + 1}/{totalEpochs} ---")
    
    epochLoss = 0
    for step, batch in enumerate(trainLoader):
        batch = {k: v.to(centralizedModel.device) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = centralizedModel(**batch)
            loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        epochLoss += loss.item()
        
        if (step + 1) % 50 == 0:
            print(f"   Step {step + 1}/{len(trainLoader)} | Loss: {loss.item():.4f}")
            
        del outputs, loss, batch
            
    avgTrainLoss = epochLoss / len(trainLoader)
    peakVramRecorded = getPeakReservedMemory()
    print(f" [Báo cáo Epoch {epoch + 1}] Avg Loss: {avgTrainLoss:.4f}")
    
    # Đánh giá sau mỗi Epoch
    latestEvalResults = evaluateModel(centralizedModel, testDataset, processor)
    roundAccuracy = latestEvalResults["overallAccuracy"] * 100
    roundAccClosed = latestEvalResults.get("Closed", {}).get("ACC", 0) * 100
    roundF1Closed = latestEvalResults.get("Closed", {}).get("F1", 0) * 100
    roundBleu = latestEvalResults.get("Open", {}).get("BLEU-1", 0)
    
    performanceLog.append({
        "Epoch": epoch + 1,
        "Loss": round(avgTrainLoss, 4),
        "Overall Acc (%)": round(roundAccuracy, 2),
        "Closed Acc (%)": round(roundAccClosed, 2),
        "Closed F1 (%)": round(roundF1Closed, 2),
        "Open BLEU-1": round(roundBleu, 4)
    })
    print(f" -> Overall Acc: {roundAccuracy:.2f}% | Closed Acc: {roundAccClosed:.2f}% | Open BLEU-1: {roundBleu:.4f}")
    clearMemory()

# ==========================================
# 6. TỔNG HỢP VÀ IN BÁO CÁO 
# ==========================================
totalTime = time.time() - startTotalTime

print("\n" + "="*80)
print(f" FINAL SUMMARY REPORT: CENTRALIZED BASELINE (SLAKE ENGLISH, LORA, {totalEpochs} EPOCHS) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Trainable Parameters   : {trainableParams:,} (~{trainPct:.3f}%)")
print(f"  + Total Training Time    : {totalTime:.2f} giây ({totalEpochs} Epochs)")
print(f"  + Peak Reserved Memory   : {peakVramRecorded}")
print(f"  + Average Inference Time : {latestEvalResults['avgInferenceTime']:.4f} giây/ảnh")

print("\n[2] CLINICAL EVALUATION METRICS (TẠI EPOCH CUỐI)")
if "Closed" in latestEvalResults:
    print(f"  - CLOSED QUESTIONS ({latestEvalResults['Closed']['count']} samples):")
    print(f"    * Accuracy : {latestEvalResults['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score (Weighted): {latestEvalResults['Closed']['F1']*100:.2f}%")
if "Open" in latestEvalResults:
    print(f"  - OPEN QUESTIONS ({latestEvalResults['Open']['count']} samples):")
    print(f"    * BLEU-1   : {latestEvalResults['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {latestEvalResults['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {latestEvalResults['Open']['BERTScore']:.4f}")

print("\n[3] TRAINING LOG")
evalTable = pd.DataFrame(performanceLog)
print(evalTable.to_string(index=False))

print("\n" + "="*80)
print(" QUÁ TRÌNH HUẤN LUYỆN TẬP TRUNG ĐÃ HOÀN TẤT THÀNH CÔNG ")
print("="*80)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


--- KHỞI TẠO HẠ TẦNG CENTRALIZED LORA ---


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

... (30 outputs left out)

 -> Overall Acc: 72.10% | Closed Acc: 79.09% | Open BLEU-1: 0.6491

--- EPOCH 2/2 ---
   Step 50/2460 | Loss: 0.1618
   Step 100/2460 | Loss: 0.0024
   Step 150/2460 | Loss: 0.1120
   Step 200/2460 | Loss: 0.0058
   Step 250/2460 | Loss: 1.0054
   Step 300/2460 | Loss: 0.1687
   Step 350/2460 | Loss: 0.3443
   Step 400/2460 | Loss: 0.5207
   Step 450/2460 | Loss: 0.0722
   Step 500/2460 | Loss: 0.0601
   Step 550/2460 | Loss: 0.2891
   Step 600/2460 | Loss: 0.2903
   Step 650/2460 | Loss: 0.1801
   Step 700/2460 | Loss: 0.0159
   Step 750/2460 | Loss: 0.0361
   Step 800/2460 | Loss: 0.0306
   Step 850/2460 | Loss: 0.1085
   Step 900/2460 | Loss: 0.0029
   Step 950/2460 | Loss: 0.1707
   Step 1000/2460 | Loss: 0.1129
   Step 1050/2460 | Loss: 0.0728
   Step 1100/2460 | Loss: 0.0258
   Step 1150/2460 | Loss: 0.0043
   Step 1200/2460 | Loss: 0.0022
   Step 1250/2460 | Loss: 0.1351
   Step 1300/2460 | Loss: 0.0046
   Step 1350/2460 | Loss: 0.0323
   Step 1400/2460 | Loss: 0.1091
   Step 14

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 -> Overall Acc: 75.31% | Closed Acc: 81.73% | Open BLEU-1: 0.7169

 FINAL SUMMARY REPORT: CENTRALIZED BASELINE (SLAKE ENGLISH, LORA, 2 EPOCHS) 

[1] PERFORMANCE & HARDWARE METRICS
  + Trainable Parameters   : 36,929,536 (~1.644%)
  + Total Training Time    : 12266.22 giây (2 Epochs)
  + Peak Reserved Memory   : 13.22 GB
  + Average Inference Time : 0.5215 giây/ảnh

[2] CLINICAL EVALUATION METRICS (TẠI EPOCH CUỐI)
  - CLOSED QUESTIONS (416 samples):
    * Accuracy : 81.73%
    * F1-Score (Weighted): 81.42%
  - OPEN QUESTIONS (645 samples):
    * BLEU-1   : 0.7169
    * ROUGE-L  : 0.7753
    * BERTScore: 0.9729

[3] TRAINING LOG
 Epoch   Loss  Overall Acc (%)  Closed Acc (%)  Closed F1 (%)  Open BLEU-1
     1 0.3290            72.10           79.09          78.29       0.6491
     2 0.1886            75.31           81.73          81.42       0.7169

 QUÁ TRÌNH HUẤN LUYỆN TẬP TRUNG ĐÃ HOÀN TẤT THÀNH CÔNG 
